In [ ]:
# Cell 1: Imports and Setup
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Load datasets
df_donors = pd.read_csv("donors.csv")
df_donations = pd.read_csv("donations.csv")
df_campaigns = pd.read_csv("campaigns.csv")

df_donations["donation_date"] = pd.to_datetime(df_donations["donation_date"])

In [ ]:
# Cell 2: Donors Summary & Location Chart
print(f"Total Donors: {len(df_donors):,}")
print(f"Average Single Donation: ${df_donations['amount'].mean():,.2f}")
print(f"Median Single Donation: ${df_donations['amount'].median():,.2f}")

plt.figure(figsize=(10, 4))
loc_counts = df_donors["location"].value_counts().reset_index()
loc_counts.columns = ["location", "count"]
sns.barplot(data=loc_counts, x="location", y="count", palette="viridis")
plt.title("Donors by Location", fontsize=12, fontweight="bold")
plt.xlabel("State / Location")
plt.ylabel("Donor Count")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Donations Summary & Time Series Charts
print(f"Total Donations: {len(df_donations):,}")
print(f"Total Donation Value: ${df_donations['amount'].sum():,.2f}")

# Monthly Trend Line Chart
df_donations["year_month"] = (
    df_donations["donation_date"].dt.to_period("M").dt.to_timestamp()
)
monthly_trend = (
    df_donations.groupby("year_month")["amount"].sum().reset_index()
)

plt.figure(figsize=(12, 4))
sns.lineplot(
    data=monthly_trend,
    x="year_month",
    y="amount",
    color="teal",
    marker="o",
    linewidth=2,
)
plt.title("Monthly Donation Revenue Trend", fontsize=12, fontweight="bold")
plt.ylabel("Total Revenue ($)")
plt.xlabel("Month")
plt.gca().yaxis.set_major_formatter("${x:,.0f}")
plt.tight_layout()
plt.show()

# Annual Trend Dual-Axis Chart
df_donations["year"] = df_donations["donation_date"].dt.year
annual_summary = (
    df_donations[df_donations["year"] < 2026]
    .groupby("year")
    .agg(total_revenue=("amount", "sum"), total_count=("amount", "count"))
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=annual_summary, x="year", y="total_revenue", ax=ax1, color="steelblue"
)
ax1.set_ylabel("Total Revenue ($)", color="steelblue", fontweight="bold")
ax1.yaxis.set_major_formatter("${x:,.0f}")

ax2 = ax1.twinx()
sns.lineplot(
    data=annual_summary,
    x=annual_summary.index,
    y="total_count",
    ax=ax2,
    color="darkorange",
    marker="s",
    linewidth=2.5,
)
ax2.set_ylabel("Donation Volume", color="darkorange", fontweight="bold")
ax2.grid(False)

plt.title(
    "Annual Growth: Revenue & Volume (2021 - 2025)",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Top Campaigns Analysis & Visualizations
campaign_metrics = (
    df_donations.groupby("campaign_id")
    .agg(
        total_revenue=("amount", "sum"),
        donor_count=("donor_id", "nunique"),
        donation_count=("amount", "count"),
    )
    .reset_index()
)

campaign_summary = campaign_metrics.merge(
    df_campaigns[["campaign_id", "campaign_name", "campaign_type"]],
    on="campaign_id",
)

# Top 10 by Revenue
top_rev = campaign_summary.sort_values(
    by="total_revenue", ascending=False
).head(10)
plt.figure(figsize=(10, 4))
sns.barplot(
    data=top_rev, y="campaign_name", x="total_revenue", palette="Blues_r"
)
plt.title("Top 10 Campaigns by Revenue", fontsize=12, fontweight="bold")
plt.xlabel("Total Raised ($)")
plt.gca().xaxis.set_major_formatter("${x:,.0f}")
plt.tight_layout()
plt.show()

# Top 10 by Donors
top_donors = campaign_summary.sort_values(
    by="donor_count", ascending=False
).head(10)
plt.figure(figsize=(10, 4))
sns.barplot(
    data=top_donors, y="campaign_name", x="donor_count", palette="Greens_r"
)
plt.title("Top 10 Campaigns by Unique Donors", fontsize=12, fontweight="bold")
plt.xlabel("Number of Donors")
plt.tight_layout()
plt.show()